# UCI HAR Dataset — Federated Learning Data Pipeline

This notebook downloads, preprocesses, and partitions the **UCI Human Activity Recognition (HAR)** dataset for simulated Federated Learning experiments.

| Property | Value |
|---|---|
| **Primary task** | Activity recognition (6 classes) |
| **Sensitive attribute** | Subject identity (30 classes) |
| **FL setup** | Each client = one subject (person) |
| **Features** | 561 time/frequency-domain sensor features |

## 1. Imports & Configuration

In [1]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple, Dict, List
from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch version : {torch.__version__}")
print(f"NumPy  version  : {np.__version__}")

PyTorch version : 2.11.0+cpu
NumPy  version  : 2.3.5


In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
DATA_ROOT = Path("data")
RAW_DIR   = DATA_ROOT / "raw"
PROC_DIR  = DATA_ROOT / "processed"

# ── Dataset constants ──────────────────────────────────────────────────────────
ACTIVITY_LABELS = {
    1: "WALKING", 2: "WALKING_UPSTAIRS", 3: "WALKING_DOWNSTAIRS",
    4: "SITTING", 5: "STANDING", 6: "LAYING"
}
N_FEATURES = 561
N_SUBJECTS = 30
N_CLASSES  = 6

print("Config OK")
print(f"  RAW_DIR  : {RAW_DIR.resolve()}")
print(f"  PROC_DIR : {PROC_DIR.resolve()}")

Config OK
  RAW_DIR  : C:\Users\PMLS\Desktop\federated\data\raw
  PROC_DIR : C:\Users\PMLS\Desktop\federated\data\processed


## 2. Download & Extract Dataset

In [3]:
def download_har(force: bool = False) -> Path:
    """Download and extract the UCI HAR dataset if not already present."""
    zip_path = RAW_DIR / "UCI_HAR.zip"
    har_dir  = RAW_DIR / "UCI HAR Dataset"

    RAW_DIR.mkdir(parents=True, exist_ok=True)

    if har_dir.exists() and not force:
        print(f"[data] Dataset already at {har_dir}")
        return har_dir

    print("[data] Downloading UCI HAR Dataset (~60 MB)…")
    urllib.request.urlretrieve(DATA_URL, zip_path)
    print("[data] Extracting…")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(RAW_DIR)
    print("[data] Done.")
    return har_dir

In [4]:
# Download (skipped automatically if already present)
har_dir = download_har(force=False)
print(f"\nDataset path: {har_dir}")

[data] Downloading UCI HAR Dataset (~60 MB)…
[data] Extracting…
[data] Done.

Dataset path: data\raw\UCI HAR Dataset


## 3. Raw Data Loading

In [5]:
def _load_split(har_dir: Path, split: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load one split (train / test).

    Returns
    -------
    X       : (N, 561) float32  — sensor feature vectors
    y_act   : (N,)     int64    — 0-indexed activity labels
    y_subj  : (N,)     int64    — 0-indexed subject IDs
    """
    base   = har_dir / split
    X      = np.loadtxt(base / f"X_{split}.txt",       dtype=np.float32)
    y_act  = np.loadtxt(base / f"y_{split}.txt",       dtype=np.int64) - 1  # 0-index
    y_subj = np.loadtxt(base / f"subject_{split}.txt", dtype=np.int64) - 1
    return X, y_act, y_subj

In [6]:
print("Loading train split…")
X_train_raw, y_act_train_raw, y_subj_train_raw = _load_split(har_dir, "train")

print("Loading test split…")
X_test_raw, y_act_test_raw, y_subj_test_raw = _load_split(har_dir, "test")

print(f"\nTrain shape : X={X_train_raw.shape}  y_act={y_act_train_raw.shape}  y_subj={y_subj_train_raw.shape}")
print(f"Test  shape : X={X_test_raw.shape}   y_act={y_act_test_raw.shape}   y_subj={y_subj_test_raw.shape}")
print(f"\nTrain subjects : {np.unique(y_subj_train_raw)}")
print(f"Test  subjects : {np.unique(y_subj_test_raw)}")
print(f"\nActivity classes : {np.unique(y_act_train_raw)}  → {list(ACTIVITY_LABELS.values())}")

Loading train split…
Loading test split…

Train shape : X=(7352, 561)  y_act=(7352,)  y_subj=(7352,)
Test  shape : X=(2947, 561)   y_act=(2947,)   y_subj=(2947,)

Train subjects : [ 0  2  4  5  6  7 10 13 14 15 16 18 20 21 22 24 25 26 27 28 29]
Test  subjects : [ 1  3  8  9 11 12 17 19 23]

Activity classes : [0 1 2 3 4 5]  → ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']


## 4. Preprocessing (Standardisation)

In [7]:
def load_and_preprocess(har_dir: Path = None) -> Dict:
    """
    Load both splits, fit a StandardScaler on train, apply to test.

    Returns dict with keys:
        X_train, y_act_train, y_subj_train,
        X_test,  y_act_test,  y_subj_test,
        scaler
    """
    if har_dir is None:
        har_dir = RAW_DIR / "UCI HAR Dataset"
    if not har_dir.exists():
        har_dir = download_har()

    X_tr, ya_tr, ys_tr = _load_split(har_dir, "train")
    X_te, ya_te, ys_te = _load_split(har_dir, "test")

    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr).astype(np.float32)
    X_te   = scaler.transform(X_te).astype(np.float32)

    return {
        "X_train": X_tr, "y_act_train": ya_tr, "y_subj_train": ys_tr,
        "X_test":  X_te, "y_act_test":  ya_te, "y_subj_test":  ys_te,
        "scaler":  scaler,
    }

In [8]:
data = load_and_preprocess(har_dir)

print(f"Train : {data['X_train'].shape}  mean≈{data['X_train'].mean():.4f}  std≈{data['X_train'].std():.4f}")
print(f"Test  : {data['X_test'].shape}   mean≈{data['X_test'].mean():.4f}  std≈{data['X_test'].std():.4f}")

Train : (7352, 561)  mean≈0.0000  std≈1.0000
Test  : (2947, 561)   mean≈-0.0111  std≈0.9289


## 5. Federated Partitioning (one client per subject)

In [9]:
def partition_by_subject(data: Dict) -> Dict[int, Dict]:
    """
    Split training data into one partition per subject (FL client).

    Returns
    -------
    { subject_id (0-29) : {"X": ..., "y_act": ..., "y_subj": ...} }
    """
    partitions = {}
    X, ya, ys = data["X_train"], data["y_act_train"], data["y_subj_train"]

    for sid in np.unique(ys):
        mask = ys == sid
        partitions[int(sid)] = {
            "X":      X[mask],
            "y_act":  ya[mask],
            "y_subj": ys[mask],
        }
    return partitions

In [10]:
partitions = partition_by_subject(data)

print(f"Total FL clients : {len(partitions)}\n")
print(f"{'Client':>8}  {'Samples':>8}  {'Activities'}")
print("-" * 40)
for sid, part in sorted(partitions.items()):
    acts = np.unique(part["y_act"])
    print(f"{sid:>8}  {len(part['y_act']):>8}  {acts}")

sizes = [len(p["y_act"]) for p in partitions.values()]
print(f"\nSamples/client — min={min(sizes)}  max={max(sizes)}  mean={np.mean(sizes):.0f}")

Total FL clients : 21

  Client   Samples  Activities
----------------------------------------
       0       347  [0 1 2 3 4 5]
       2       341  [0 1 2 3 4 5]
       4       302  [0 1 2 3 4 5]
       5       325  [0 1 2 3 4 5]
       6       308  [0 1 2 3 4 5]
       7       281  [0 1 2 3 4 5]
      10       316  [0 1 2 3 4 5]
      13       323  [0 1 2 3 4 5]
      14       328  [0 1 2 3 4 5]
      15       366  [0 1 2 3 4 5]
      16       368  [0 1 2 3 4 5]
      18       360  [0 1 2 3 4 5]
      20       408  [0 1 2 3 4 5]
      21       321  [0 1 2 3 4 5]
      22       372  [0 1 2 3 4 5]
      24       409  [0 1 2 3 4 5]
      25       392  [0 1 2 3 4 5]
      26       376  [0 1 2 3 4 5]
      27       382  [0 1 2 3 4 5]
      28       344  [0 1 2 3 4 5]
      29       383  [0 1 2 3 4 5]

Samples/client — min=281  max=409  mean=350


## 6. PyTorch Dataset & DataLoaders

In [11]:
class HARDataset(Dataset):
    """Thin wrapper so partitions plug straight into DataLoader."""

    def __init__(self, X: np.ndarray, y_act: np.ndarray, y_subj: np.ndarray):
        self.X      = torch.tensor(X,      dtype=torch.float32)
        self.y_act  = torch.tensor(y_act,  dtype=torch.long)
        self.y_subj = torch.tensor(y_subj, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_act[idx], self.y_subj[idx]

In [12]:
def make_client_loaders(
    partitions: Dict[int, Dict],
    batch_size: int = 32,
    shuffle:    bool = True,
) -> Dict[int, DataLoader]:
    """Create one DataLoader per FL client."""
    loaders = {}
    for cid, part in partitions.items():
        ds = HARDataset(part["X"], part["y_act"], part["y_subj"])
        loaders[cid] = DataLoader(ds, batch_size=batch_size, shuffle=shuffle)
    return loaders


def make_test_loader(data: Dict, batch_size: int = 256) -> DataLoader:
    """Create a DataLoader for the global test set."""
    ds = HARDataset(data["X_test"], data["y_act_test"], data["y_subj_test"])
    return DataLoader(ds, batch_size=batch_size, shuffle=False)

In [13]:
client_loaders = make_client_loaders(partitions, batch_size=32, shuffle=True)
test_loader    = make_test_loader(data, batch_size=256)

print(f"Client loaders : {len(client_loaders)}")
print(f"Test  loader   : {len(test_loader)} batches")

Client loaders : 21
Test  loader   : 12 batches


## 7. Sanity Check

In [14]:
# Inspect one batch from client 0
X_batch, ya_batch, ys_batch = next(iter(client_loaders[0]))

print("Batch from client 0")
print(f"  X      : {X_batch.shape}   dtype={X_batch.dtype}")
print(f"  y_act  : {ya_batch.shape}  dtype={ya_batch.dtype}  unique={ya_batch.unique().tolist()}")
print(f"  y_subj : {ys_batch.shape}  dtype={ys_batch.dtype}  unique={ys_batch.unique().tolist()}")

# Inspect one batch from the test loader
X_t, ya_t, ys_t = next(iter(test_loader))
print(f"\nBatch from test loader")
print(f"  X      : {X_t.shape}")
print(f"  y_act  : {ya_t.shape}")
print(f"  y_subj : {ys_t.shape}")

print("\n✅ Data pipeline OK — ready for federated training.")

Batch from client 0
  X      : torch.Size([32, 561])   dtype=torch.float32
  y_act  : torch.Size([32])  dtype=torch.int64  unique=[0, 1, 2, 3, 4, 5]
  y_subj : torch.Size([32])  dtype=torch.int64  unique=[0]

Batch from test loader
  X      : torch.Size([256, 561])
  y_act  : torch.Size([256])
  y_subj : torch.Size([256])

✅ Data pipeline OK — ready for federated training.


## 8. Dataset Summary

In [15]:
summary = pd.DataFrame([
    {
        "Client": sid,
        "Samples": len(part["y_act"]),
        "Unique Activities": len(np.unique(part["y_act"])),
        "Activity IDs": str(np.unique(part["y_act"].tolist() if hasattr(part["y_act"], "tolist") else part["y_act"])),
    }
    for sid, part in sorted(partitions.items())
])

print(summary.to_string(index=False))
print(f"\nTotal train samples : {summary['Samples'].sum()}")
print(f"Total test  samples : {len(data['y_act_test'])}")

 Client  Samples  Unique Activities  Activity IDs
      0      347                  6 [0 1 2 3 4 5]
      2      341                  6 [0 1 2 3 4 5]
      4      302                  6 [0 1 2 3 4 5]
      5      325                  6 [0 1 2 3 4 5]
      6      308                  6 [0 1 2 3 4 5]
      7      281                  6 [0 1 2 3 4 5]
     10      316                  6 [0 1 2 3 4 5]
     13      323                  6 [0 1 2 3 4 5]
     14      328                  6 [0 1 2 3 4 5]
     15      366                  6 [0 1 2 3 4 5]
     16      368                  6 [0 1 2 3 4 5]
     18      360                  6 [0 1 2 3 4 5]
     20      408                  6 [0 1 2 3 4 5]
     21      321                  6 [0 1 2 3 4 5]
     22      372                  6 [0 1 2 3 4 5]
     24      409                  6 [0 1 2 3 4 5]
     25      392                  6 [0 1 2 3 4 5]
     26      376                  6 [0 1 2 3 4 5]
     27      382                  6 [0 1 2 3 4 5]
